In [1]:
import numpy as np
import pandas as pd
import os
import random
import copy

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss

# 固定配置
SEEDS = [114514, 2024, 3407, 42, 123, 777, 2025, 8888]
BATCH_SIZE = 1024
EPOCHS = 50
LR = 2e-3
AUX_LOSS_WEIGHT = 0.5
FEATURE_DROPOUT = 0.1
N_FOLDS = 5
PROB_EPS = 1e-15
SUBMISSION_PATH = "submission.csv"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Device:", device)
print("Seeds:", SEEDS)

PyTorch: 2.14.1+cu130 | CUDA: 13.0
Device: cuda
Seeds: [114514, 2024, 3407, 42, 123, 777, 2025, 8888]


In [2]:
train_features = pd.read_csv("./data/train_features.csv")
test_features = pd.read_csv("./data/test_features.csv")
train_drug = pd.read_csv("./data/train_drug.csv")
train_targets_scored = pd.read_csv("./data/train_targets_scored.csv")
train_targets_nonscored = pd.read_csv("./data/train_targets_nonscored.csv")
sample_submission = pd.read_csv("./data/sample_submission.csv")

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)
print("Scored targets:", train_targets_scored.shape)
print("Non-scored targets:", train_targets_nonscored.shape)
print("Submission template:", sample_submission.shape)

Train features: (23814, 876)
Test features: (3982, 876)
Scored targets: (23814, 207)
Non-scored targets: (23814, 403)
Submission template: (3982, 207)


In [ ]:
target_columns = [c for c in train_targets_scored.columns if c != "sig_id"]
y = train_targets_scored[target_columns].values.astype(np.float32)

train_ids = train_features["sig_id"].copy()
test_ids = test_features["sig_id"].copy()

aux_columns = [c for c in train_targets_nonscored.columns if c != "sig_id"]
aux_aligned = (
    train_targets_nonscored
    .set_index("sig_id")
    .loc[train_ids.values, aux_columns]
)
assert (aux_aligned.index.values == train_ids.values).all()
y_aux = aux_aligned.values.astype(np.float32)

groups = (
    train_features[["sig_id"]]
    .merge(train_drug, on="sig_id", how="left", validate="one_to_one")["drug_id"]
    .values
)

# 特征编码
X_train = train_features.drop(columns=["sig_id"]).copy()
X_test = test_features.drop(columns=["sig_id"]).copy()
all_features = pd.concat([X_train, X_test], axis=0, ignore_index=True)

categorical_columns = ["cp_type", "cp_time", "cp_dose"]
for col in categorical_columns:
    all_features[col] = all_features[col].astype(str)

all_features = pd.get_dummies(
    all_features, columns=categorical_columns, dtype=np.float32
)

X_train = all_features.iloc[:len(train_features)].copy()
X_test = all_features.iloc[len(train_features):].copy()

numeric_cols = [c for c in X_train.columns if c.startswith(("g-", "c-"))]
num_idx = [X_train.columns.get_loc(c) for c in numeric_cols]

print("Encoded train:", X_train.shape)
print("Encoded test:", X_test.shape)
print("Scored labels:", y.shape, "| Aux labels:", y_aux.shape)
print("Numeric features:", len(numeric_cols))

Encoded train: (23814, 879)
Encoded test: (3982, 879)
Scored labels: (23814, 206) | Aux labels: (23814, 402)
Numeric features: 872


In [ ]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_aux_tensor = torch.tensor(y_aux, dtype=torch.float32)

input_dim = X_train.shape[1]
output_dim = len(target_columns)
aux_dim = y_aux_tensor.shape[1]

# feature dropout
num_mask = torch.zeros(input_dim, device=device)
num_mask[torch.tensor(num_idx, dtype=torch.long, device=device)] = 1.0


class MoAMultiTaskMLP(nn.Module):
    def __init__(self, input_dim, output_dim, aux_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.main_head = nn.Linear(512, output_dim)
        self.aux_head = nn.Linear(512, aux_dim)

    def forward(self, x):
        h = self.backbone(x)
        return self.main_head(h), self.aux_head(h)

In [ ]:
train_ctl_mask = (train_features["cp_type"] == "ctl_vehicle").values
test_ctl_mask = (test_features["cp_type"] == "ctl_vehicle").values


def competition_log_loss(y_true, y_pred, ctl_mask):
    p = np.clip(y_pred, PROB_EPS, 1 - PROB_EPS).copy()
    p[ctl_mask] = PROB_EPS
    losses = [
        log_loss(y_true[:, j], p[:, j], labels=[0, 1])
        for j in range(y_true.shape[1])
    ]
    return float(np.mean(losses))


gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_tensor, y_train_tensor, groups=groups))

oof_log_sum = np.zeros((len(X_train), output_dim), dtype=np.float64)
test_log_sum = np.zeros((len(X_test), output_dim), dtype=np.float64)

for seed in SEEDS:
    print(f"\n########## Seed {seed} ##########")
    seed_oof = np.zeros((len(X_train), output_dim), dtype=np.float64)
    seed_test = np.zeros((len(X_test), output_dim), dtype=np.float64)

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"========== Seed {seed} | Fold {fold+1}/{N_FOLDS} ==========")
        set_seed(seed + fold)

        # 标准化
        X_tr = X_train_tensor[train_idx].clone()
        X_val = X_train_tensor[val_idx].clone()
        X_test_scaled = X_test_tensor.clone()

        scaler = StandardScaler()
        X_tr_np = X_tr.numpy().copy()
        X_val_np = X_val.numpy().copy()
        X_test_np = X_test_scaled.numpy().copy()

        X_tr_np[:, num_idx] = scaler.fit_transform(X_tr_np[:, num_idx])
        X_val_np[:, num_idx] = scaler.transform(X_val_np[:, num_idx])
        X_test_np[:, num_idx] = scaler.transform(X_test_np[:, num_idx])

        X_tr = torch.from_numpy(X_tr_np)
        X_val = torch.from_numpy(X_val_np)
        X_test_scaled = torch.from_numpy(X_test_np)

        y_tr = y_train_tensor[train_idx]
        y_val = y_train_tensor[val_idx]
        y_aux_tr = y_aux_tensor[train_idx]

        train_loader = DataLoader(
            TensorDataset(X_tr, y_tr, y_aux_tr),
            batch_size=BATCH_SIZE,
            shuffle=True,
            drop_last=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_val, y_val),
            batch_size=BATCH_SIZE * 2,
            shuffle=False,
        )

        model = MoAMultiTaskMLP(input_dim, output_dim, aux_dim).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )

        best_val = float("inf")
        best_state = copy.deepcopy(model.state_dict())

        for epoch in range(EPOCHS):
            model.train()
            for batch_X, batch_y, batch_y_aux in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                batch_y_aux = batch_y_aux.to(device)

                keep = (torch.rand_like(batch_X) > FEATURE_DROPOUT).float()
                scale = keep / (1.0 - FEATURE_DROPOUT)
                scale = torch.where(
                    num_mask.bool(), scale, torch.ones_like(scale)
                )
                batch_X = batch_X * scale

                optimizer.zero_grad()
                out_main, out_aux = model(batch_X)
                loss = (
                    criterion(out_main, batch_y)
                    + AUX_LOSS_WEIGHT * criterion(out_aux, batch_y_aux)
                )
                loss.backward()
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X = batch_X.to(device)
                    batch_y = batch_y.to(device)
                    out_main, _ = model(batch_X)
                    val_loss += criterion(out_main, batch_y).item() * batch_X.size(0)
            val_loss /= len(val_loader.dataset)
            scheduler.step(val_loss)

            if val_loss < best_val:
                best_val = val_loss
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        model.eval()

        with torch.no_grad():
            out_val, _ = model(X_val.to(device))
            seed_oof[val_idx] = torch.sigmoid(out_val).cpu().numpy()

            out_test, _ = model(X_test_scaled.to(device))
            seed_test += torch.sigmoid(out_test).cpu().numpy() / N_FOLDS

        print(f"Seed {seed} | Fold {fold+1} Best Val: {best_val:.5f}")

    oof_log_sum += np.log(np.clip(seed_oof, PROB_EPS, 1 - PROB_EPS))
    test_log_sum += np.log(np.clip(seed_test, PROB_EPS, 1 - PROB_EPS))
    print(f"Seed {seed} OOF: {competition_log_loss(y, seed_oof, train_ctl_mask):.6f}")

# 几何平均集成
oof_preds = np.exp(oof_log_sum / len(SEEDS))
test_preds = np.exp(test_log_sum / len(SEEDS))

cv_score = competition_log_loss(y, oof_preds, train_ctl_mask)
print(f"\nFinal geometric ensemble OOF: {cv_score:.6f}")

# 对照组预测置为极小值
test_preds[test_ctl_mask, :] = PROB_EPS

submission = pd.DataFrame(test_preds, columns=target_columns)
submission.insert(0, "sig_id", test_ids)
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print(submission.head())


########## Seed 114514 ##########
========== Seed 114514 | Fold 1/5 ==========
Seed 114514 | Fold 1 Best Val: 0.01310
========== Seed 114514 | Fold 2/5 ==========
Seed 114514 | Fold 2 Best Val: 0.02234
========== Seed 114514 | Fold 3/5 ==========
Seed 114514 | Fold 3 Best Val: 0.02315
========== Seed 114514 | Fold 4/5 ==========
Seed 114514 | Fold 4 Best Val: 0.01721
========== Seed 114514 | Fold 5/5 ==========
Seed 114514 | Fold 5 Best Val: 0.01871
Seed 114514 OOF: 0.018620

########## Seed 2024 ##########
========== Seed 2024 | Fold 1/5 ==========
Seed 2024 | Fold 1 Best Val: 0.01308
========== Seed 2024 | Fold 2/5 ==========
Seed 2024 | Fold 2 Best Val: 0.02199
========== Seed 2024 | Fold 3/5 ==========
Seed 2024 | Fold 3 Best Val: 0.02292
========== Seed 2024 | Fold 4/5 ==========
Seed 2024 | Fold 4 Best Val: 0.01754
========== Seed 2024 | Fold 5/5 ==========
Seed 2024 | Fold 5 Best Val: 0.01869
Seed 2024 OOF: 0.018579

########## Seed 3407 ##########
========== Seed 3407 | Fold 1